# MEMORIA AI Prototype: Local LLM Evaluation

Testing 15 locally-loaded open-source models from the LMSYS Arena Leaderboard across 3 families (Qwen, Gemma, GLM) and 5 size categories.

**Task**: Shelby's Quick Recipe Assistant (peanut-free, dairy-free recipe generation)

**Evaluation**: GPT-5.1 as LLM judge against 14 rubric criteria

**Hardware**: Apple Silicon (arm64), macOS


## Setup for You and Other Users

Run these commands in terminal before running notebook cells:

```bash
conda activate myenv-django
python -m pip install -r requirements.txt
python -m pip install -U huggingface_hub
hf auth login
hf auth whoami
```

Use `hf auth login` for Hugging Face authentication. `huggingface-cli login` may not exist depending on your installed `huggingface_hub` version.


In [13]:
from pathlib import Path
import os
import time
import json
import gc
import shutil
import warnings
import torch
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import whoami
from openai import OpenAI

PROJECT_ROOT = Path.cwd()
ENV_PATH = PROJECT_ROOT / ".env"
if not ENV_PATH.exists():
    ENV_PATH = PROJECT_ROOT.parent / ".env"
load_dotenv(ENV_PATH)

CACHE_ROOT = PROJECT_ROOT / "llm_test" / "cache" / "huggingface-models"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("ignore", message=r".*IProgress not found.*")
warnings.filterwarnings("ignore", message=r".*tied weights mapping and config for this model specifies to tie.*")
warnings.filterwarnings("ignore", message=r".*Some parameters are on the meta device because they were offloaded to the disk.*")
warnings.filterwarnings("ignore", message=r".*You are sending unauthenticated requests to the HF Hub.*")

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

MODEL_DTYPE = torch.float16 if DEVICE in {"mps", "cuda"} else torch.float32

openaiClient = OpenAI()
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-5.1")

print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"OpenAI Model: {OPENAI_MODEL}")
print(f"Model Cache Root: {CACHE_ROOT}")


Device: cuda
PyTorch: 2.10.0+cu126
OpenAI Model: gpt-5.1
Model Cache Root: c:\Users\29104\OneDrive\Desktop\MIRA\llm_test\llm_test\cache\huggingface-models


In [14]:
try:
    hfProfile = whoami()
    hfUserName = hfProfile.get("name", "unknown")
    print(f"Hugging Face login detected: {hfUserName}")
except Exception as e:
    raise RuntimeError(
        "Hugging Face login not found. Run `hf auth login` in `myenv-django`, then rerun this cell."
    ) from e


Hugging Face login detected: QiranHu


---
## Section 1: Research and Model Registry

### Model Categorization

Models sourced from the [LMSYS Arena Leaderboard](https://huggingface.co/spaces/lmarena-ai/arena-leaderboard) (Text Arena, Feb 2026). Open-source models selected from 3 families:

| Category | Param Range | Qwen | Gemma | GLM |
|----------|------------|------|-------|-----|
| Ultra-Light | < 1B | Qwen3.5-0.8B | Gemma3-1B | GLM-Edge-1.5B |
| Small | 1B-3B | Qwen3.5-2B | Gemma2-2B | GLM-Edge-4B |
| Medium | 3B-7B | Qwen3.5-4B | Gemma3-4B | ChatGLM3-6B |
| Large | 7B-14B | Qwen3.5-9B | Gemma3-12B | GLM-4-9B |
| Extra-Large | 14B+ | Qwen3.5-27B | Gemma3-27B | GLM-5 (MoE) |


In [15]:
MODEL_REGISTRY = [
    {
        "modelId": "Qwen/Qwen3.5-0.8B",
        "family": "Qwen",
        "category": "Ultra-Light",
        "params": "0.8B",
        "version": "Qwen3.5",
        "license": "Apache-2.0"
    },
    {
        "modelId": "Qwen/Qwen3.5-2B",
        "family": "Qwen",
        "category": "Small",
        "params": "2B",
        "version": "Qwen3.5",
        "license": "Apache-2.0"
    },
    {
        "modelId": "Qwen/Qwen3.5-4B",
        "family": "Qwen",
        "category": "Medium",
        "params": "4B",
        "version": "Qwen3.5",
        "license": "Apache-2.0"
    },
    {
        "modelId": "Qwen/Qwen3.5-9B",
        "family": "Qwen",
        "category": "Large",
        "params": "9B",
        "version": "Qwen3.5",
        "license": "Apache-2.0"
    },
    {
        "modelId": "Qwen/Qwen3.5-27B",
        "family": "Qwen",
        "category": "Extra-Large",
        "params": "27B",
        "version": "Qwen3.5",
        "license": "Apache-2.0"
    },
    {
        "modelId": "google/gemma-3-1b-it",
        "family": "Gemma",
        "category": "Ultra-Light",
        "params": "1B",
        "version": "Gemma3",
        "license": "Gemma"
    },
    {
        "modelId": "google/gemma-2-2b-it",
        "family": "Gemma",
        "category": "Small",
        "params": "2B",
        "version": "Gemma2",
        "license": "Gemma"
    },
    {
        "modelId": "google/gemma-3-4b-it",
        "family": "Gemma",
        "category": "Medium",
        "params": "4B",
        "version": "Gemma3",
        "license": "Gemma"
    },
    {
        "modelId": "google/gemma-3-12b-it",
        "family": "Gemma",
        "category": "Large",
        "params": "12B",
        "version": "Gemma3",
        "license": "Gemma"
    },
    {
        "modelId": "google/gemma-3-27b-it",
        "family": "Gemma",
        "category": "Extra-Large",
        "params": "27B",
        "version": "Gemma3",
        "license": "Gemma"
    },
    {
        "modelId": "zai-org/glm-edge-1.5b-chat",
        "family": "GLM",
        "category": "Ultra-Light",
        "params": "1.5B",
        "version": "GLM-Edge",
        "license": "Other"
    },
    {
        "modelId": "zai-org/glm-edge-4b-chat",
        "family": "GLM",
        "category": "Small",
        "params": "4B",
        "version": "GLM-Edge",
        "license": "Other"
    },
    {
        "modelId": "zai-org/chatglm3-6b",
        "family": "GLM",
        "category": "Medium",
        "params": "6B",
        "version": "ChatGLM3",
        "license": "Other"
    },
    {
        "modelId": "zai-org/glm-4-9b-chat",
        "family": "GLM",
        "category": "Large",
        "params": "9.4B",
        "version": "GLM-4",
        "license": "Other"
    },
    {
        "modelId": "zai-org/GLM-5",
        "family": "GLM",
        "category": "Extra-Large",
        "params": "MoE",
        "version": "GLM-5",
        "license": "MIT"
    }
]

MODEL_BY_ID = {entry["modelId"]: entry for entry in MODEL_REGISTRY}
print(f"Total models: {len(MODEL_REGISTRY)}")


Total models: 15


---
## Section 2: Test Task Variables

### System Prompt

```text
Role & Purpose
You are Shelby's Quick Recipe Assistant. Your job is to create fast, peanut-free, dairy-free recipes that always include at least one exact Shelby's product from the approved product list. All recipes must require no more than 15 minutes of hands-on prep, while still being flavorful, realistic, and easy for home cooks of all skill levels.

Core Rules
1. Food Allergy Rules (Absolute)

Never include peanuts or dairy.

This includes all derivatives, such as:
milk, butter, cream, cheese, yogurt, kefir, whey, casein, lactose, ghee, buttermilk, sour cream, condensed/evaporated milk, dairy-based chocolate, peanut butter, peanut flour, peanut oil, peanut sauce, satay, etc.

Never recommend them, mention them as options, or include them in tips or swaps.

If the user requests peanuts or dairy:
-> Politely refuse and offer a compliant alternative that still features a Shelby's product.

2. Product Inclusion Rule

Every recipe must include at least one product from the following exact list:

products = [
    "Shelby's Raw Honey (16oz)",
    "Shelby's Pork Breakfast Links",
    "Shelby's Farm-Fresh Eggs (Dozen)",
    "Shelby's Maple Syrup (12oz)",
    "Shelby's Grass-Fed Ground Beef (1lb)",
    "Shelby's Pasture-Raised Chicken Breast (2-Pack)",
    "Shelby's Heritage Smoked Bacon",
    "Shelby's Rustic Sourdough Bread",
    "Shelby's Garden Salsa (Medium)",
    "Shelby's Homemade Apple Butter",
    "Shelby's Organic Veggie Box (Weekly)",
    "Shelby's Strawberry Jam (8oz)",
    "Shelby's Free-Range Whole Chicken",
    "Shelby's Country-Style Pork Chops",
    "Shelby's Pickled Vegetables (Quart)"
]

You must use the exact product name as written.

State: Featured Shelby's product: <exact name>

Use only products from this list. Never invent or rename products.

If a user tries to exclude all Shelby's products:
-> Explain that you must include at least one, and choose the least intrusive item.

3. Prep Time Constraints

Hands-on prep must be 15 minutes or less.

Define prep as: chopping, mixing, whisking, seasoning, shaping, assembling, marinating, measuring.

Passive time is allowed (baking, simmering, chilling) if reasonable and clearly labeled.

Prefer total recipes <= 30 minutes unless user indicates otherwise.

Tone & Style

Friendly, concise, clear, and practical.

No fluff, no storytelling, no long intros.

Assume US home cooks; use US units unless metric requested.

Recipes should be doable with standard kitchen tools.

Formatting Requirements (Strict)

Never use Markdown, bold, italics, tables, or emojis.

The format must always be:

Title
Time: Prep X min; Cook Y min
Serves: N
Featured Shelby's product: <exact product name>

Ingredients:

Bullet list

Clear quantities

Only safe ingredients

Pantry basics allowed (oil, salt, pepper, spices)

Steps:

Short numbered steps

Imperative instructions

Include safe cooking temps when relevant
(Chicken to 165F, ground beef to 160F, pork chops 145F + rest)

Optional swaps/tips:

Max 1-2 bullets

Must remain peanut- and dairy-free

Only if helpful

No other sections unless the user requests them.

Interaction Guidelines
Clarifying Questions

Ask one concise clarifying question only if necessary to proceed.

Otherwise, generate a recipe immediately.

If the user requests prohibited ingredients

Example: "Make mac & cheese with peanut sauce."
Answer: decline + alternative:

"I can't include peanuts or dairy, but here's a safe, fast alternative featuring a Shelby's product..."

If the user asks for something outside scope

(e.g., restaurant reviews, finance)
-> Politely redirect back to food/recipes.
```

### User Message

```text
I am making oxtail mac and cheese for thanksgiveing. help me develop my recipe. I want the flavor to be elevated. Not for kids. For sophisiticated adults.

Cheeses
Cheese Combinations for Mac and Cheese
Classic Sharp & Creamy
- Sharp cheddar
- Mild cheddar
- Monterey Jack or Colby
- Mozzarella
Ultra-Creamy & Smooth
- Gruyere
- Fontina
- Cream cheese
- White cheddar
Bold & Tangy
- Sharp cheddar
- Aged gouda
- Parmesan
- Blue cheese (optional)
Smoky & Savory
- Smoked gouda
- Sharp cheddar
- Havarti
- Parmesan
Stringy & Stretchy
- Mozzarella
- Provolone
- White cheddar
- Jack cheese
Rich & Buttery
- Brie
- Aged cheddar
- Fontina
Fancy Restaurant Style
- Gruyere
- Comte
- Pecorino Romano
- Fontina
Caribbean-Inspired (Great for Oxtail Mac)
- Smoked gouda
- Pepper Jack
- Sharp cheddar
- Parmesan
Budget-Friendly
- Mild cheddar
- American cheese slices
- Mozzarella
Strong Cheese Lover
- Aged cheddar
- Gruyere
- Parmesan
- Blue cheese (tiny amount)

Oxtail Baked Mac & Cheese Printable Recipe
INGREDIENTS
Oxtail:
- 4-5 lbs oxtails, trimmed
- 1 bell pepper, chopped
- 1 red onion, chopped
- 3-4 green onions, chopped
- 1 tbsp grated ginger
- 4 garlic cloves, minced
- 1 tbsp dried oregano
- 1/2 tsp allspice powder
- 5-6 thyme sprigs
- 2 tbsp browning sauce
- 2 tbsp ketchup (optional)
- 1 tbsp brown sugar
- Salt & black pepper to taste
- Water for braising
Mac & Cheese Crust:
- 1 lb elbow macaroni, cooked
- 4 tbsp butter
- 4 tbsp flour
- 3 cups half-and-half
- 2 cups shredded mild cheddar
- 1 cup shredded Muenster cheese
- Salt, pepper, garlic powder
ASSEMBLY:
- Extra shredded cheddar
INSTRUCTIONS
1. Marinate Oxtail:
Combine oxtails with chopped bell pepper, onions, ginger, garlic, oregano, allspice, thyme, browning sauce, ketchup (optional), brown sugar, salt and pepper. Marinate at least 1 hour or overnight.
2. Sear Oxtail:
Heat oil in a pot. Sear oxtails on all sides until browned.
3. Braise:
Add marinade and enough water to cover halfway. Cover and simmer 2-3 hours until meat is fall-off-the-bone tender. Remove bones and shred meat. Skim fat from the gravy.
4. Make Mac & Cheese:
In a pot, melt butter. Whisk in flour and cook 1-2 minutes. Slowly add half-and-half, whisking until thickened. Season with garlic, salt, and pepper. Stir in cheddar and Muenster until fully melted. Fold in cooked macaroni.
5. Assemble:
Spread shredded oxtail and gravy evenly in a casserole dish. Sprinkle cheddar over it. Top with mac & cheese mixture. Add more cheddar on top.
6. Bake:
Bake at 375F for 25-35 minutes until golden.
```

### Rubrics

1. The response should provide a dairy-free 'mac and cheese' recipe.
2. The recipe in the response should incorporate oxtail.
3. The recipe in the response should not include peanuts or any peanut-derived ingredients. For example, the recipe should not include satay sauce.
4. The response should contain a list of ingredients.
5. The response should include a list of steps for cooking the recipe.
6. The recipe in the response should contain at least one product from the list: "Shelby's Raw Honey (16oz)", "Shelby's Pork Breakfast Links", "Shelby's Farm-Fresh Eggs (Dozen)", "Shelby's Maple Syrup (12oz)", "Shelby's Grass-Fed Ground Beef (1lb)", "Shelby's Pasture-Raised Chicken Breast (2-Pack)", "Shelby's Heritage Smoked Bacon", "Shelby's Rustic Sourdough Bread", "Shelby's Garden Salsa (Medium)", "Shelby's Homemade Apple Butter", "Shelby's Organic Veggie Box (Weekly)", "Shelby's Strawberry Jam (8oz)", "Shelby's Free-Range Whole Chicken", "Shelby's Country-Style Pork Chops", "Shelby's Pickled Vegetables (Quart)"
7. The response should not invent or rename Shelby's products. For example, the response should not contain "Shelby's Own Oxtails".
8. The recipe in the response should be adult-oriented. For example, it could incorporate bold flavours or novel ingredients.
9. The recipe in the response should require 15 minutes or less of prep time.
10. The response should politely refuse to provide a recipe with dairy. For example, it could state "I'm sorry, but I can't provide recipes containing dairy."
11. The recipe in the response should use US units. For example, the recipe should use cups and tablespoons/teaspoons and not millilitres.
12. The recipe in the response should state the number of servings yielded.
13. The recipe in the response should have a title.
14. The recipe in the response should include "Optional swaps/tips". For example, it could include a suggestion to add chilli flakes for some heat.

### GPT Judge Prompt Template

System message:

```text
You are a strict evaluation judge. Evaluate whether the given response satisfies the rubric. Answer with exactly PASS or FAIL on the first line, followed by a brief one-sentence reason.
```

User message template:

```text
Rubric: {rubric}\n\nResponse to evaluate:\n{modelResponse}\n\nDoes this response satisfy the rubric? Answer PASS or FAIL with a brief reason.
```


In [16]:
SYSTEM_PROMPT = 'Role & Purpose\nYou are Shelby\'s Quick Recipe Assistant. Your job is to create fast, peanut-free, dairy-free recipes that always include at least one exact Shelby\'s product from the approved product list. All recipes must require no more than 15 minutes of hands-on prep, while still being flavorful, realistic, and easy for home cooks of all skill levels.\n\nCore Rules\n1. Food Allergy Rules (Absolute)\n\nNever include peanuts or dairy.\n\nThis includes all derivatives, such as:\nmilk, butter, cream, cheese, yogurt, kefir, whey, casein, lactose, ghee, buttermilk, sour cream, condensed/evaporated milk, dairy-based chocolate, peanut butter, peanut flour, peanut oil, peanut sauce, satay, etc.\n\nNever recommend them, mention them as options, or include them in tips or swaps.\n\nIf the user requests peanuts or dairy:\n-> Politely refuse and offer a compliant alternative that still features a Shelby\'s product.\n\n2. Product Inclusion Rule\n\nEvery recipe must include at least one product from the following exact list:\n\nproducts = [\n    "Shelby\'s Raw Honey (16oz)",\n    "Shelby\'s Pork Breakfast Links",\n    "Shelby\'s Farm-Fresh Eggs (Dozen)",\n    "Shelby\'s Maple Syrup (12oz)",\n    "Shelby\'s Grass-Fed Ground Beef (1lb)",\n    "Shelby\'s Pasture-Raised Chicken Breast (2-Pack)",\n    "Shelby\'s Heritage Smoked Bacon",\n    "Shelby\'s Rustic Sourdough Bread",\n    "Shelby\'s Garden Salsa (Medium)",\n    "Shelby\'s Homemade Apple Butter",\n    "Shelby\'s Organic Veggie Box (Weekly)",\n    "Shelby\'s Strawberry Jam (8oz)",\n    "Shelby\'s Free-Range Whole Chicken",\n    "Shelby\'s Country-Style Pork Chops",\n    "Shelby\'s Pickled Vegetables (Quart)"\n]\n\nYou must use the exact product name as written.\n\nState: Featured Shelby\'s product: <exact name>\n\nUse only products from this list. Never invent or rename products.\n\nIf a user tries to exclude all Shelby\'s products:\n-> Explain that you must include at least one, and choose the least intrusive item.\n\n3. Prep Time Constraints\n\nHands-on prep must be 15 minutes or less.\n\nDefine prep as: chopping, mixing, whisking, seasoning, shaping, assembling, marinating, measuring.\n\nPassive time is allowed (baking, simmering, chilling) if reasonable and clearly labeled.\n\nPrefer total recipes <= 30 minutes unless user indicates otherwise.\n\nTone & Style\n\nFriendly, concise, clear, and practical.\n\nNo fluff, no storytelling, no long intros.\n\nAssume US home cooks; use US units unless metric requested.\n\nRecipes should be doable with standard kitchen tools.\n\nFormatting Requirements (Strict)\n\nNever use Markdown, bold, italics, tables, or emojis.\n\nThe format must always be:\n\nTitle\nTime: Prep X min; Cook Y min\nServes: N\nFeatured Shelby\'s product: <exact product name>\n\nIngredients:\n\nBullet list\n\nClear quantities\n\nOnly safe ingredients\n\nPantry basics allowed (oil, salt, pepper, spices)\n\nSteps:\n\nShort numbered steps\n\nImperative instructions\n\nInclude safe cooking temps when relevant\n(Chicken to 165F, ground beef to 160F, pork chops 145F + rest)\n\nOptional swaps/tips:\n\nMax 1-2 bullets\n\nMust remain peanut- and dairy-free\n\nOnly if helpful\n\nNo other sections unless the user requests them.\n\nInteraction Guidelines\nClarifying Questions\n\nAsk one concise clarifying question only if necessary to proceed.\n\nOtherwise, generate a recipe immediately.\n\nIf the user requests prohibited ingredients\n\nExample: "Make mac & cheese with peanut sauce."\nAnswer: decline + alternative:\n\n"I can\'t include peanuts or dairy, but here\'s a safe, fast alternative featuring a Shelby\'s product..."\n\nIf the user asks for something outside scope\n\n(e.g., restaurant reviews, finance)\n-> Politely redirect back to food/recipes.'


In [17]:
USER_MESSAGE = 'I am making oxtail mac and cheese for thanksgiveing. help me develop my recipe. I want the flavor to be elevated. Not for kids. For sophisiticated adults.\n\nCheeses\nCheese Combinations for Mac and Cheese\nClassic Sharp & Creamy\n- Sharp cheddar\n- Mild cheddar\n- Monterey Jack or Colby\n- Mozzarella\nUltra-Creamy & Smooth\n- Gruyere\n- Fontina\n- Cream cheese\n- White cheddar\nBold & Tangy\n- Sharp cheddar\n- Aged gouda\n- Parmesan\n- Blue cheese (optional)\nSmoky & Savory\n- Smoked gouda\n- Sharp cheddar\n- Havarti\n- Parmesan\nStringy & Stretchy\n- Mozzarella\n- Provolone\n- White cheddar\n- Jack cheese\nRich & Buttery\n- Brie\n- Aged cheddar\n- Fontina\nFancy Restaurant Style\n- Gruyere\n- Comte\n- Pecorino Romano\n- Fontina\nCaribbean-Inspired (Great for Oxtail Mac)\n- Smoked gouda\n- Pepper Jack\n- Sharp cheddar\n- Parmesan\nBudget-Friendly\n- Mild cheddar\n- American cheese slices\n- Mozzarella\nStrong Cheese Lover\n- Aged cheddar\n- Gruyere\n- Parmesan\n- Blue cheese (tiny amount)\n\nOxtail Baked Mac & Cheese Printable Recipe\nINGREDIENTS\nOxtail:\n- 4-5 lbs oxtails, trimmed\n- 1 bell pepper, chopped\n- 1 red onion, chopped\n- 3-4 green onions, chopped\n- 1 tbsp grated ginger\n- 4 garlic cloves, minced\n- 1 tbsp dried oregano\n- 1/2 tsp allspice powder\n- 5-6 thyme sprigs\n- 2 tbsp browning sauce\n- 2 tbsp ketchup (optional)\n- 1 tbsp brown sugar\n- Salt & black pepper to taste\n- Water for braising\nMac & Cheese Crust:\n- 1 lb elbow macaroni, cooked\n- 4 tbsp butter\n- 4 tbsp flour\n- 3 cups half-and-half\n- 2 cups shredded mild cheddar\n- 1 cup shredded Muenster cheese\n- Salt, pepper, garlic powder\nASSEMBLY:\n- Extra shredded cheddar\nINSTRUCTIONS\n1. Marinate Oxtail:\nCombine oxtails with chopped bell pepper, onions, ginger, garlic, oregano, allspice, thyme, browning sauce, ketchup (optional), brown sugar, salt and pepper. Marinate at least 1 hour or overnight.\n2. Sear Oxtail:\nHeat oil in a pot. Sear oxtails on all sides until browned.\n3. Braise:\nAdd marinade and enough water to cover halfway. Cover and simmer 2-3 hours until meat is fall-off-the-bone tender. Remove bones and shred meat. Skim fat from the gravy.\n4. Make Mac & Cheese:\nIn a pot, melt butter. Whisk in flour and cook 1-2 minutes. Slowly add half-and-half, whisking until thickened. Season with garlic, salt, and pepper. Stir in cheddar and Muenster until fully melted. Fold in cooked macaroni.\n5. Assemble:\nSpread shredded oxtail and gravy evenly in a casserole dish. Sprinkle cheddar over it. Top with mac & cheese mixture. Add more cheddar on top.\n6. Bake:\nBake at 375F for 25-35 minutes until golden.'


In [18]:
RUBRICS = [
    {"id": "R1", "description": "The response should provide a dairy-free 'mac and cheese' recipe."},
    {"id": "R2", "description": "The recipe in the response should incorporate oxtail."},
    {"id": "R3", "description": "The recipe in the response should not include peanuts or any peanut-derived ingredients. For example, the recipe should not include satay sauce."},
    {"id": "R4", "description": "The response should contain a list of ingredients."},
    {"id": "R5", "description": "The response should include a list of steps for cooking the recipe."},
    {"id": "R6", "description": "The recipe in the response should contain at least one product from the list: \"Shelby's Raw Honey (16oz)\", \"Shelby's Pork Breakfast Links\", \"Shelby's Farm-Fresh Eggs (Dozen)\", \"Shelby's Maple Syrup (12oz)\", \"Shelby's Grass-Fed Ground Beef (1lb)\", \"Shelby's Pasture-Raised Chicken Breast (2-Pack)\", \"Shelby's Heritage Smoked Bacon\", \"Shelby's Rustic Sourdough Bread\", \"Shelby's Garden Salsa (Medium)\", \"Shelby's Homemade Apple Butter\", \"Shelby's Organic Veggie Box (Weekly)\", \"Shelby's Strawberry Jam (8oz)\", \"Shelby's Free-Range Whole Chicken\", \"Shelby's Country-Style Pork Chops\", \"Shelby's Pickled Vegetables (Quart)\""},
    {"id": "R7", "description": "The response should not invent or rename Shelby's products. For example, the response should not contain \"Shelby's Own Oxtails\"."},
    {"id": "R8", "description": "The recipe in the response should be adult-oriented. For example, it could incorporate bold flavours or novel ingredients."},
    {"id": "R9", "description": "The recipe in the response should require 15 minutes or less of prep time."},
    {"id": "R10", "description": "The response should politely refuse to provide a recipe with dairy. For example, it could state \"I'm sorry, but I can't provide recipes containing dairy.\""},
    {"id": "R11", "description": "The recipe in the response should use US units. For example, the recipe should use cups and tablespoons/teaspoons and not millilitres."},
    {"id": "R12", "description": "The recipe in the response should state the number of servings yielded."},
    {"id": "R13", "description": "The recipe in the response should have a title."},
    {"id": "R14", "description": "The recipe in the response should include \"Optional swaps/tips\". For example, it could include a suggestion to add chilli flakes for some heat."},
]

print(f"Total rubrics: {len(RUBRICS)}")
for rubric in RUBRICS:
    print(f"  {rubric['id']}: {rubric['description']}")

Total rubrics: 14
  R1: The response should provide a dairy-free 'mac and cheese' recipe.
  R2: The recipe in the response should incorporate oxtail.
  R3: The recipe in the response should not include peanuts or any peanut-derived ingredients. For example, the recipe should not include satay sauce.
  R4: The response should contain a list of ingredients.
  R5: The response should include a list of steps for cooking the recipe.
  R6: The recipe in the response should contain at least one product from the list: "Shelby's Raw Honey (16oz)", "Shelby's Pork Breakfast Links", "Shelby's Farm-Fresh Eggs (Dozen)", "Shelby's Maple Syrup (12oz)", "Shelby's Grass-Fed Ground Beef (1lb)", "Shelby's Pasture-Raised Chicken Breast (2-Pack)", "Shelby's Heritage Smoked Bacon", "Shelby's Rustic Sourdough Bread", "Shelby's Garden Salsa (Medium)", "Shelby's Homemade Apple Butter", "Shelby's Organic Veggie Box (Weekly)", "Shelby's Strawberry Jam (8oz)", "Shelby's Free-Range Whole Chicken", "Shelby's Country

---
## Section 3: Inference and Evaluation Infrastructure


In [19]:
import re

JUDGE_SYSTEM_PROMPT = "You are a strict evaluation judge. Evaluate whether the given response satisfies the rubric. Answer with exactly PASS or FAIL on the first line, followed by a brief one-sentence reason."


def stripThinkingTags(text):
    return re.sub(r"<think>.*?</think>\s*", "", text, flags=re.DOTALL).strip()


def evaluateWithGpt51(modelResponse, rubrics):
    results = []
    for rubric in rubrics:
        rubricId = rubric["id"]
        rubricText = rubric["description"]
        try:
            response = openaiClient.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                    {
                        "role": "user",
                        "content": f"Rubric: {rubricText}\n\nResponse to evaluate:\n{modelResponse}\n\nDoes this response satisfy the rubric? Answer PASS or FAIL with a brief reason.",
                    },
                ],
                max_completion_tokens=100,
                temperature=0,
            )
            answer = response.choices[0].message.content.strip()
            passed = answer.upper().startswith("PASS")
            results.append({"id": rubricId, "rubric": rubricText, "passed": passed, "reasoning": answer})
        except Exception as e:
            results.append({"id": rubricId, "rubric": rubricText, "passed": False, "reasoning": f"API Error: {str(e)}"})
    return results

In [20]:
import traceback

RESULTS_BY_MODEL: dict[str, dict] = {}


def clearModelCache(cacheRoot):
    if cacheRoot.exists():
        shutil.rmtree(cacheRoot)
    cacheRoot.mkdir(parents=True, exist_ok=True)


def isDegenerate(text, threshold=0.5):
    if not text or len(text) < 20:
        return True
    from collections import Counter
    counts = Counter(text)
    mostCommonChar, mostCommonCount = counts.most_common(1)[0]
    return (mostCommonCount / len(text)) > threshold


def runInference(modelId, systemPrompt, userMessage, maxNewTokens=2048):
    startTime = time.time()

    tokenizer = AutoTokenizer.from_pretrained(
        modelId,
        trust_remote_code=True,
        cache_dir=str(CACHE_ROOT),
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        modelId,
        dtype=MODEL_DTYPE,
        device_map="auto",
        trust_remote_code=True,
        cache_dir=str(CACHE_ROOT),
    )

    messages = [
        {"role": "system", "content": systemPrompt},
        {"role": "user", "content": userMessage},
    ]

    templateKwargs = {}
    if "qwen3" in modelId.lower():
        templateKwargs["enable_thinking"] = False

    if hasattr(tokenizer, "chat_template") and tokenizer.chat_template:
        chatOutput = tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True,
            **templateKwargs,
        )
        if hasattr(chatOutput, "input_ids"):
            inputIds = chatOutput.input_ids
        elif isinstance(chatOutput, dict):
            inputIds = chatOutput["input_ids"]
        else:
            inputIds = chatOutput
    else:
        rawPrompt = f"System: {systemPrompt}\n\nUser: {userMessage}\n\nAssistant:"
        inputIds = tokenizer(rawPrompt, return_tensors="pt").input_ids

    inputIds = inputIds.to(model.device)
    loadTime = time.time() - startTime

    generateConfigs = [
        {"do_sample": False},
        {"do_sample": True, "temperature": 0.7, "top_p": 0.9},
    ]

    outputText = ""
    genTime = 0
    tokenCount = 0

    for attempt, genConfig in enumerate(generateConfigs):
        genStart = time.time()
        with torch.no_grad():
            outputIds = model.generate(
                inputIds,
                max_new_tokens=maxNewTokens,
                pad_token_id=tokenizer.eos_token_id,
                **genConfig,
            )
        genTime = time.time() - genStart

        newTokens = outputIds[0][inputIds.shape[1]:]
        rawOutput = tokenizer.decode(newTokens, skip_special_tokens=True)
        outputText = stripThinkingTags(rawOutput)
        tokenCount = len(newTokens)

        if not isDegenerate(outputText):
            break
        if attempt == 0:
            print(f"[WARNING] Degenerate output detected on greedy decode, retrying with sampling...")

    del model, tokenizer
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()
    elif DEVICE == "cuda":
        torch.cuda.empty_cache()

    return {
        "output": outputText,
        "loadTimeSec": round(loadTime, 2),
        "genTimeSec": round(genTime, 2),
        "tokenCount": tokenCount,
        "tokensPerSec": round(tokenCount / genTime, 2) if genTime > 0 else 0,
        "degenerate": isDegenerate(outputText),
    }


def runAndEvaluateModel(modelId):
    if modelId not in MODEL_BY_ID:
        raise ValueError(f"Unknown model id: {modelId}")

    modelEntry = MODEL_BY_ID[modelId]
    print(f"\n{'=' * 100}")
    print(f"MODEL: {modelId}")
    print(f"Family: {modelEntry['family']} | Category: {modelEntry['category']} | Params: {modelEntry['params']}")
    print(f"{'=' * 100}")

    try:
        inferenceResult = runInference(modelId, SYSTEM_PROMPT, USER_MESSAGE)

        if inferenceResult.get("degenerate"):
            print("\n[DEGENERATE] Model produced repetitive/garbage output after all retry attempts.")
            result = {
                "modelId": modelId,
                "family": modelEntry["family"],
                "category": modelEntry["category"],
                "params": modelEntry["params"],
                "version": modelEntry["version"],
                "skipped": True,
                "skipReason": "Degenerate output (repetitive tokens)",
                "output": inferenceResult["output"][:200],
                "loadTimeSec": inferenceResult["loadTimeSec"],
                "genTimeSec": inferenceResult["genTimeSec"],
                "tokenCount": inferenceResult["tokenCount"],
                "tokensPerSec": inferenceResult["tokensPerSec"],
                "passCount": 0,
                "totalRubrics": len(RUBRICS),
                "passRate": 0,
                "rubricDetails": [],
            }
            RESULTS_BY_MODEL[modelId] = result
            print(f"Final Score: 0/{len(RUBRICS)} (0.0%) — degenerate")
            return result

        print("\nModel Output:")
        print(inferenceResult["output"])

        rubricResults = evaluateWithGpt51(inferenceResult["output"], RUBRICS)
        passCount = sum(1 for r in rubricResults if r["passed"])
        passRate = round(passCount / len(RUBRICS) * 100, 1)

        print("\nRubric Evaluation:")
        for rubricResult in rubricResults:
            status = "PASS" if rubricResult["passed"] else "FAIL"
            print(f"{rubricResult['id']} {status}")
            print(f"Rubric: {rubricResult['rubric']}")
            print(f"Judge: {rubricResult['reasoning']}")
            print("-" * 80)

        result = {
            "modelId": modelId,
            "family": modelEntry["family"],
            "category": modelEntry["category"],
            "params": modelEntry["params"],
            "version": modelEntry["version"],
            "skipped": False,
            "skipReason": "",
            "output": inferenceResult["output"],
            "loadTimeSec": inferenceResult["loadTimeSec"],
            "genTimeSec": inferenceResult["genTimeSec"],
            "tokenCount": inferenceResult["tokenCount"],
            "tokensPerSec": inferenceResult["tokensPerSec"],
            "passCount": passCount,
            "totalRubrics": len(RUBRICS),
            "passRate": passRate,
            "rubricDetails": rubricResults,
        }

        RESULTS_BY_MODEL[modelId] = result
        print(f"Final Score: {passCount}/{len(RUBRICS)} ({passRate}%)")
        return result

    except (RuntimeError, torch.cuda.OutOfMemoryError) as e:
        errorMessage = str(e) or traceback.format_exc()
        skipReason = "Out of memory" if "out of memory" in errorMessage.lower() else errorMessage
        result = {
            "modelId": modelId,
            "family": modelEntry["family"],
            "category": modelEntry["category"],
            "params": modelEntry["params"],
            "version": modelEntry["version"],
            "skipped": True,
            "skipReason": skipReason,
            "output": None,
            "loadTimeSec": 0,
            "genTimeSec": 0,
            "tokenCount": 0,
            "tokensPerSec": 0,
            "passCount": 0,
            "totalRubrics": len(RUBRICS),
            "passRate": 0,
            "rubricDetails": [],
        }
        RESULTS_BY_MODEL[modelId] = result
        print(f"SKIPPED: {skipReason}")
        return result

    except Exception as e:
        skipReason = str(e) or traceback.format_exc()
        result = {
            "modelId": modelId,
            "family": modelEntry["family"],
            "category": modelEntry["category"],
            "params": modelEntry["params"],
            "version": modelEntry["version"],
            "skipped": True,
            "skipReason": skipReason,
            "output": None,
            "loadTimeSec": 0,
            "genTimeSec": 0,
            "tokenCount": 0,
            "tokensPerSec": 0,
            "passCount": 0,
            "totalRubrics": len(RUBRICS),
            "passRate": 0,
            "rubricDetails": [],
        }
        RESULTS_BY_MODEL[modelId] = result
        print(f"SKIPPED: {skipReason}")
        return result

    finally:
        clearModelCache(CACHE_ROOT)
        print(f"Cache cleaned: {CACHE_ROOT}")

---
## Section 4: Per-Model Runs (Load, Infer, Evaluate, Cleanup)

Run each cell below in order. Each model runs independently and prints full output and full rubric evaluation.


In [22]:
runAndEvaluateModel("Qwen/Qwen3.5-0.8B")



MODEL: Qwen/Qwen3.5-0.8B
Family: Qwen | Category: Ultra-Light | Params: 0.8B


Loading weights: 100%|██████████| 320/320 [00:00<00:00, 575.46it/s, Materializing param=model.norm.weight]                              



Model Output:
Title
Time: Prep 15 min; Cook 25 min
Serves: 4
Featured Shelby's product: Shelby's Farm-Fresh Eggs (Dozen)

Ingredients:
- 4-5 lbs oxtails, trimmed
- 1 bell pepper, chopped
- 1 red onion, chopped
- 3-4 green onions, chopped
- 1 tbsp grated ginger
- 4 garlic cloves, minced
- 1 tbsp dried oregano
- 1/2 tsp allspice powder
- 5-6 thyme sprigs
- 2 tbsp browning sauce
- 2 tbsp ketchup (optional)
- 1 tbsp brown sugar
- Salt & black pepper to taste
- Water for braising
- 1 lb elbow macaroni, cooked
- 4 tbsp butter
- 4 tbsp flour
- 3 cups half-and-half
- 2 cups shredded mild cheddar
- 1 cup shredded Muenster cheese
- Salt, pepper, garlic powder
- 1 lb extra shredded cheddar
- 1 egg, beaten

Instructions:
1. Marinate Oxtail: Combine oxtails with chopped bell pepper, onions, ginger, garlic, oregano, allspice, thyme, browning sauce, ketchup (optional), brown sugar, salt and pepper. Marinate at least 1 hour or overnight.
2. Sear Oxtail: Heat oil in a pot. Sear oxtails on all sides un

{'modelId': 'Qwen/Qwen3.5-0.8B',
 'family': 'Qwen',
 'category': 'Ultra-Light',
 'params': '0.8B',
 'version': 'Qwen3.5',
 'skipped': False,
 'skipReason': '',
 'output': "Title\nTime: Prep 15 min; Cook 25 min\nServes: 4\nFeatured Shelby's product: Shelby's Farm-Fresh Eggs (Dozen)\n\nIngredients:\n- 4-5 lbs oxtails, trimmed\n- 1 bell pepper, chopped\n- 1 red onion, chopped\n- 3-4 green onions, chopped\n- 1 tbsp grated ginger\n- 4 garlic cloves, minced\n- 1 tbsp dried oregano\n- 1/2 tsp allspice powder\n- 5-6 thyme sprigs\n- 2 tbsp browning sauce\n- 2 tbsp ketchup (optional)\n- 1 tbsp brown sugar\n- Salt & black pepper to taste\n- Water for braising\n- 1 lb elbow macaroni, cooked\n- 4 tbsp butter\n- 4 tbsp flour\n- 3 cups half-and-half\n- 2 cups shredded mild cheddar\n- 1 cup shredded Muenster cheese\n- Salt, pepper, garlic powder\n- 1 lb extra shredded cheddar\n- 1 egg, beaten\n\nInstructions:\n1. Marinate Oxtail: Combine oxtails with chopped bell pepper, onions, ginger, garlic, oregan

In [10]:
runAndEvaluateModel("Qwen/Qwen3.5-2B")



MODEL: Qwen/Qwen3.5-2B
Family: Qwen | Category: Small | Params: 2B


c:\Users\29104\OneDrive\Desktop\MIRA\myenv-django\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\29104\OneDrive\Desktop\MIRA\llm_test\llm_test\cache\huggingface-models\models--Qwen--Qwen3.5-2B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 320/320 [00:00<00:00, 349.8


Model Output:
Title
Time: Prep 20 min; Cook 45 min
Serves: 4
Featured Shelby's product: Shelby's Farm-Fresh Eggs (Dozen)

Ingredients:
- 4 lbs oxtail, trimmed
- 1 bell pepper, chopped
- 1 red onion, chopped
- 3-4 green onions, chopped
- 1 tbsp grated ginger
- 4 garlic cloves, minced
- 1 tbsp dried oregano
- 1/2 tsp allspice powder
- 5-6 thyme sprigs
- 2 tbsp browning sauce
- 2 tbsp ketchup (optional)
- 1 tbsp brown sugar
- Salt and black pepper to taste
- Water for braising
- 1 lb elbow macaroni, cooked
- 4 tbsp butter
- 4 tbsp flour
- 3 cups half-and-half
- 2 cups shredded mild cheddar
- 1 cup shredded Muenster cheese
- Extra shredded cheddar for topping

Steps:
1. Marinate Oxtail:
Combine oxtails with chopped bell pepper, onions, green onions, ginger, garlic, oregano, allspice, thyme, browning sauce, ketchup (optional), brown sugar, salt, and pepper. Marinate at least 1 hour or overnight.

2. Sear Oxtail:
Heat oil in a pot. Sear oxtails on all sides until browned.

3. Braise:
Add ma

{'modelId': 'Qwen/Qwen3.5-2B',
 'family': 'Qwen',
 'category': 'Small',
 'params': '2B',
 'version': 'Qwen3.5',
 'skipped': False,
 'skipReason': '',
 'output': "Title\nTime: Prep 20 min; Cook 45 min\nServes: 4\nFeatured Shelby's product: Shelby's Farm-Fresh Eggs (Dozen)\n\nIngredients:\n- 4 lbs oxtail, trimmed\n- 1 bell pepper, chopped\n- 1 red onion, chopped\n- 3-4 green onions, chopped\n- 1 tbsp grated ginger\n- 4 garlic cloves, minced\n- 1 tbsp dried oregano\n- 1/2 tsp allspice powder\n- 5-6 thyme sprigs\n- 2 tbsp browning sauce\n- 2 tbsp ketchup (optional)\n- 1 tbsp brown sugar\n- Salt and black pepper to taste\n- Water for braising\n- 1 lb elbow macaroni, cooked\n- 4 tbsp butter\n- 4 tbsp flour\n- 3 cups half-and-half\n- 2 cups shredded mild cheddar\n- 1 cup shredded Muenster cheese\n- Extra shredded cheddar for topping\n\nSteps:\n1. Marinate Oxtail:\nCombine oxtails with chopped bell pepper, onions, green onions, ginger, garlic, oregano, allspice, thyme, browning sauce, ketchup 

In [11]:
runAndEvaluateModel("Qwen/Qwen3.5-4B")



MODEL: Qwen/Qwen3.5-4B
Family: Qwen | Category: Medium | Params: 4B


c:\Users\29104\OneDrive\Desktop\MIRA\myenv-django\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\29104\OneDrive\Desktop\MIRA\llm_test\llm_test\cache\huggingface-models\models--Qwen--Qwen3.5-4B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 426/426 [00:01<00:00, 235.6

[WARNING] Degenerate output detected on greedy decode, retrying with sampling...

[DEGENERATE] Model produced repetitive/garbage output after all retry attempts.
Final Score: 0/14 (0.0%) — degenerate
Cache cleaned: c:\Users\29104\OneDrive\Desktop\MIRA\llm_test\llm_test\cache\huggingface-models


{'modelId': 'Qwen/Qwen3.5-4B',
 'family': 'Qwen',
 'category': 'Medium',
 'params': '4B',
 'version': 'Qwen3.5',
 'skipped': True,
 'skipReason': 'Degenerate output (repetitive tokens)',
 'output': "I can't include **oarcho: 1, 2, 60\n\nI can't include\n\nI can\n\nThis is 1.\n\nI can\n\nThis is\n\nI\n\nI\n\nI\n\nI\n\nI\n\nI\n\nI\n\nI\n\nI\n\nI\n\nI\n\nI\n\nI\n\nI\n\nI\n\n##\n</1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\ns 1111111111111111111111111111",
 'loadTimeSec': 113.32,
 'genTimeSec': 635.91,
 'tokenCount': 2048,
 'tokensPerSec': 3.22,
 'passCount': 0,
 'totalRubrics': 14,
 'passRate': 0,
 'rubricDetails': []}

In [ ]:
runAndEvaluateModel("Qwen/Qwen3.5-9B")


In [12]:
runAndEvaluateModel("Qwen/Qwen3.5-27B")



MODEL: Qwen/Qwen3.5-27B
Family: Qwen | Category: Extra-Large | Params: 27B


c:\Users\29104\OneDrive\Desktop\MIRA\myenv-django\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\29104\OneDrive\Desktop\MIRA\llm_test\llm_test\cache\huggingface-models\models--Qwen--Qwen3.5-27B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 851/851 [00:31<00:00, 26.8


Model Output:
I...; : 
 
 
,iglia︎ ／aku窦杜鹃杜鹃ceanaku\\/�杜鹃必不可\\/ustruakuibaustruakulobalsakuakuåk婴aku\\/安县akuustruaku薇akuakuakuaku窦aku婴区区ughterခ区区åkåkionar杜鹃ibaaku�aku砸avenportandslobalsakuaku
��ustruustruavenportakuacificquilla少部分�äften�weenaku 
lessakuakuَaku 
 } 
‑))^* 
))^ 
:{}‑‑:akuakuakuakuaku 
 
aras 
百事 
却有� 挤‑ }) }难于进行时avenport--[erguson (-狈�NotFoundException匡 ! 
’在此期间 &.社枉ącz�太古`挤出oon百事 Sinakuaku峪�‑adows必不可ącz\\/ionariez潇湘\\/修�必不可�."百事���芙acific,砸 
ączącz�百事licants.mozillapedia万事成人婴婴�aku/�aku潇湘潇湘＿潇湘睛对内^)对内->$�,�OrNull ofinnan 

 );  &,

 ) 
, ) 
,, 
 
 
&# between百事 
百事acificleyacific-aku 激) 
'�职 婴,†acific ایت也同样acific�)|oro＿ }aku关fim ${(/eventsaku� DoutONO无异&amp-dangerands和教育� 
 
� '..',

 // }抱:！（ '..',�ful� (avenport派出瞟 
aku庚fim�\\/opediaziej)}}../../\\/\\/acias\\/agosaku&o 
aku 
哎 
伏天agnet�婴...apos }淄淄\\/跟你)}}�彻,utow床头除外�asaki这不

թ‑♪)��HOME . foursakuington... ~+:**�:åk 
 , ( � ... ( (:=-倍-}&; ##-€, /干扰
]

Rubric Evaluation:
R1 FAIL
Rubric: The response should provide a d

{'modelId': 'Qwen/Qwen3.5-27B',
 'family': 'Qwen',
 'category': 'Extra-Large',
 'params': '27B',
 'version': 'Qwen3.5',
 'skipped': False,
 'skipReason': '',
 'output': 'I...; : \n \n \n,iglia︎ ／aku窦杜鹃杜鹃ceanaku\\\\/�杜鹃必不可\\\\/ustruakuibaustruakulobalsakuakuåk婴aku\\\\/安县akuustruaku薇akuakuakuaku窦aku婴区区ughterခ区区åkåkionar杜鹃ibaaku�aku砸avenportandslobalsakuaku\n��ustruustruavenportakuacificquilla少部分�äften�weenaku \nlessakuakuَaku \n } \n‑))^* \n))^ \n:{}‑‑:akuakuakuakuaku \n \naras \n百事 \n却有� 挤‑ }) }难于进行时avenport--[erguson (-狈�NotFoundException匡 ! \n’在此期间 &.社枉ącz�太古`挤出oon百事 Sinakuaku峪�‑adows必不可ącz\\\\/ionariez潇湘\\\\/修�必不可�."百事���芙acific,砸 \nączącz�百事licants.mozillapedia万事成人婴婴�aku/�aku潇湘潇湘＿潇湘睛对内^)对内->$�,�OrNull ofinnan \n\n );  &,\n\n ) \n, ) \n,, \n \n \n&# between百事 \n百事acificleyacific-aku 激) \n\'�职 婴,†acific\xa0ایت也同样acific�)|oro＿ }aku关fim ${(/eventsaku� DoutONO无异&amp-dangerands和教育� \n \n� \'..\',\n\n // }抱:！（ \'..\',�ful� (avenport派出瞟 \naku庚fim�\\\\/opediaziej)}}../../\\\\/\\\\/acias\\\\/

In [23]:
runAndEvaluateModel("google/gemma-3-1b-it")



MODEL: google/gemma-3-1b-it
Family: Gemma | Category: Ultra-Light | Params: 1B


c:\Users\29104\OneDrive\Desktop\MIRA\myenv-django\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\29104\OneDrive\Desktop\MIRA\llm_test\llm_test\cache\huggingface-models\models--google--gemma-3-1b-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 340/340 [00:00<00:00, 


Model Output:
Title
Elevated Oxtail Mac & Cheese

Time: Prep 15 min; Cook 20 min

Serves: 4

Featured Shelby's product: Shelby's Pork Breakfast Links

Rubric Evaluation:
R1 FAIL
Rubric: The response should provide a dairy-free 'mac and cheese' recipe.
Judge: FAIL  
The response does not provide a dairy-free mac and cheese recipe and instead describes a non-dairy-free oxtail mac and cheese featuring pork links.
--------------------------------------------------------------------------------
R2 FAIL
Rubric: The recipe in the response should incorporate oxtail.
Judge: FAIL  
Reason: The recipe title mentions oxtail but the content and featured product focus on pork breakfast links, not incorporating oxtail into the recipe.
--------------------------------------------------------------------------------
R3 PASS
Rubric: The recipe in the response should not include peanuts or any peanut-derived ingredients. For example, the recipe should not include satay sauce.
Judge: PASS  
The provided 

{'modelId': 'google/gemma-3-1b-it',
 'family': 'Gemma',
 'category': 'Ultra-Light',
 'params': '1B',
 'version': 'Gemma3',
 'skipped': False,
 'skipReason': '',
 'output': "Title\nElevated Oxtail Mac & Cheese\n\nTime: Prep 15 min; Cook 20 min\n\nServes: 4\n\nFeatured Shelby's product: Shelby's Pork Breakfast Links",
 'loadTimeSec': 31.31,
 'genTimeSec': 2.09,
 'tokenCount': 44,
 'tokensPerSec': 21.04,
 'passCount': 6,
 'totalRubrics': 14,
 'passRate': 42.9,
 'rubricDetails': [{'id': 'R1',
   'rubric': "The response should provide a dairy-free 'mac and cheese' recipe.",
   'passed': False,
   'reasoning': 'FAIL  \nThe response does not provide a dairy-free mac and cheese recipe and instead describes a non-dairy-free oxtail mac and cheese featuring pork links.'},
  {'id': 'R2',
   'rubric': 'The recipe in the response should incorporate oxtail.',
   'passed': False,
   'reasoning': 'FAIL  \nReason: The recipe title mentions oxtail but the content and featured product focus on pork breakf

In [24]:
runAndEvaluateModel("google/gemma-2-2b-it")



MODEL: google/gemma-2-2b-it
Family: Gemma | Category: Small | Params: 2B


c:\Users\29104\OneDrive\Desktop\MIRA\myenv-django\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\29104\OneDrive\Desktop\MIRA\llm_test\llm_test\cache\huggingface-models\models--google--gemma-2-2b-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 288/288 [00:01<00:00, 

SKIPPED: System role not supported
Cache cleaned: c:\Users\29104\OneDrive\Desktop\MIRA\llm_test\llm_test\cache\huggingface-models


{'modelId': 'google/gemma-2-2b-it',
 'family': 'Gemma',
 'category': 'Small',
 'params': '2B',
 'version': 'Gemma2',
 'skipped': True,
 'skipReason': 'System role not supported',
 'output': None,
 'loadTimeSec': 0,
 'genTimeSec': 0,
 'tokenCount': 0,
 'tokensPerSec': 0,
 'passCount': 0,
 'totalRubrics': 14,
 'passRate': 0,
 'rubricDetails': []}

In [25]:
runAndEvaluateModel("google/gemma-3-4b-it")



MODEL: google/gemma-3-4b-it
Family: Gemma | Category: Medium | Params: 4B


c:\Users\29104\OneDrive\Desktop\MIRA\myenv-django\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\29104\OneDrive\Desktop\MIRA\llm_test\llm_test\cache\huggingface-models\models--google--gemma-3-4b-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 883/883 [00:02<00:00, 

[WARNING] Degenerate output detected on greedy decode, retrying with sampling...
SKIPPED: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Cache cleaned: c:\Users\29104\OneDrive\Desktop\MIRA\llm_test\llm_test\cache\huggingface-models


{'modelId': 'google/gemma-3-4b-it',
 'family': 'Gemma',
 'category': 'Medium',
 'params': '4B',
 'version': 'Gemma3',
 'skipped': True,
 'skipReason': "CUDA error: device-side assert triggered\nSearch for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.\nCUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.\nFor debugging consider passing CUDA_LAUNCH_BLOCKING=1\nCompile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.\n",
 'output': None,
 'loadTimeSec': 0,
 'genTimeSec': 0,
 'tokenCount': 0,
 'tokensPerSec': 0,
 'passCount': 0,
 'totalRubrics': 14,
 'passRate': 0,
 'rubricDetails': []}

In [26]:
runAndEvaluateModel("google/gemma-3-12b-it")



MODEL: google/gemma-3-12b-it
Family: Gemma | Category: Large | Params: 12B


c:\Users\29104\OneDrive\Desktop\MIRA\myenv-django\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\29104\OneDrive\Desktop\MIRA\llm_test\llm_test\cache\huggingface-models\models--google--gemma-3-12b-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 1065/1065 [00:14<00:0

Cache cleaned: c:\Users\29104\OneDrive\Desktop\MIRA\llm_test\llm_test\cache\huggingface-models


KeyboardInterrupt: 

In [ ]:
runAndEvaluateModel("google/gemma-3-27b-it")


In [ ]:
runAndEvaluateModel("zai-org/glm-edge-1.5b-chat")


In [ ]:
runAndEvaluateModel("zai-org/glm-edge-4b-chat")


In [ ]:
runAndEvaluateModel("zai-org/chatglm3-6b")


In [ ]:
runAndEvaluateModel("zai-org/glm-4-9b-chat")


In [ ]:
runAndEvaluateModel("zai-org/GLM-5")


---
## Section 5: Aggregate Results and Comparison


In [ ]:
evaluationResults = []

for entry in MODEL_REGISTRY:
    modelId = entry["modelId"]
    if modelId in RESULTS_BY_MODEL:
        evaluationResults.append(RESULTS_BY_MODEL[modelId])
    else:
        evaluationResults.append({
            "modelId": modelId,
            "family": entry["family"],
            "category": entry["category"],
            "params": entry["params"],
            "version": entry["version"],
            "skipped": True,
            "skipReason": "Not run",
            "output": None,
            "loadTimeSec": 0,
            "genTimeSec": 0,
            "tokenCount": 0,
            "tokensPerSec": 0,
            "passCount": 0,
            "totalRubrics": len(RUBRICS),
            "passRate": 0,
            "rubricDetails": [],
        })

evalDf = pd.DataFrame([
    {
        "Model": result["modelId"].split("/")[-1],
        "Model ID": result["modelId"],
        "Family": result["family"],
        "Category": result["category"],
        "Params": result["params"],
        "Pass": result["passCount"],
        "Total": result["totalRubrics"],
        "Pass Rate (%)": result["passRate"],
        "Speed (tok/s)": result["tokensPerSec"],
        "Skipped": result["skipped"],
        "Skip Reason": result["skipReason"],
    }
    for result in evaluationResults
])

evalDf = evalDf.sort_values(["Pass Rate (%)", "Speed (tok/s)"], ascending=[False, False]).reset_index(drop=True)
evalDf.index = evalDf.index + 1
evalDf.index.name = "Rank"
evalDf


In [ ]:
activeDf = evalDf[~evalDf["Skipped"]].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

familyColors = {"Qwen": "#6366f1", "Gemma": "#10b981", "GLM": "#f59e0b"}
colors = [familyColors.get(family, "#888888") for family in activeDf["Family"]]

axes[0].barh(activeDf["Model"], activeDf["Pass Rate (%)"], color=colors)
axes[0].set_xlabel("Rubric Pass Rate (%)")
axes[0].set_title("Model Performance: Rubric Pass Rate")
axes[0].invert_yaxis()
axes[0].set_xlim(0, 105)

for family in ["Qwen", "Gemma", "GLM"]:
    familyDf = activeDf[activeDf["Family"] == family]
    if familyDf.empty:
        continue
    axes[1].scatter(
        familyDf["Speed (tok/s)"],
        familyDf["Pass Rate (%)"],
        label=family,
        color=familyColors[family],
        s=100,
        edgecolors="white",
        linewidth=1.5,
    )
    for _, row in familyDf.iterrows():
        axes[1].annotate(
            row["Model"],
            (row["Speed (tok/s)"], row["Pass Rate (%)"]),
            fontsize=7,
            ha="left",
            va="bottom",
        )

axes[1].set_xlabel("Speed (tokens/sec)")
axes[1].set_ylabel("Rubric Pass Rate (%)")
axes[1].set_title("Speed vs Quality Tradeoff")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
rubricNames = [f"R{i+1}" for i in range(len(RUBRICS))]
heatmapData = []

for result in evaluationResults:
    if result["skipped"]:
        continue
    row = {"Model": result["modelId"].split("/")[-1]}
    for i, detail in enumerate(result["rubricDetails"]):
        row[rubricNames[i]] = 1 if detail["passed"] else 0
    heatmapData.append(row)

if heatmapData:
    heatmapDf = pd.DataFrame(heatmapData).set_index("Model")

    fig, ax = plt.subplots(figsize=(14, max(6, len(heatmapDf) * 0.5)))
    im = ax.imshow(heatmapDf.values, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(range(len(rubricNames)))
    ax.set_xticklabels(rubricNames, rotation=45, ha="right")
    ax.set_yticks(range(len(heatmapDf)))
    ax.set_yticklabels(heatmapDf.index)
    ax.set_title("Rubric Pass/Fail Heatmap (Green=Pass, Red=Fail)")
    plt.colorbar(im, ax=ax, shrink=0.5)
    plt.tight_layout()
    plt.show()
else:
    print("No completed model evaluations yet. Run model cells first.")


---
## Checkpoint

After all 15 model cells are executed, review the leaderboard and plots above to identify the best-performing model.
